# T14 — production 275-role retained-role freeze

Run this only after PR #41 is approved and merged. It consumes the accepted T15/T16 counts and writes the frozen retained-role manifest required by the primary T17 path. The notebook fails closed if #41 is not on `main`, if the independently reviewed counts bytes changed, or if provenance no longer matches the merged PR #44 membership decision.

The resulting role set is **label-independent technical-validity retention**. It must not be described as behaviourally verified role enactment.


In [ ]:
# 0. Minimal CPU environment.
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','pytest>=8,<9'])
print(sys.version)


In [ ]:
# 1. Clone/update private repo on main.
import os, base64, getpass, subprocess, shutil
from pathlib import Path
ROOT=Path('/content/[anonymized-repository-name]')
TOKEN=os.environ.get('GITHUB_TOKEN','').strip() or getpass.getpass('GitHub token (hidden): ').strip()
AUTH=base64.b64encode(f'x-access-token:{TOKEN}'.encode()).decode()
def git(args,cwd=None,capture=False,check=True): return subprocess.run(['git','-c',f'http.extraHeader=Authorization: Basic {AUTH}',*args],cwd=str(cwd) if cwd else None,check=check,text=True,capture_output=capture)
if not (ROOT/'.git').exists():
    if ROOT.exists(): shutil.rmtree(ROOT)
    git(['clone','https://github.com/[Author-B-GitHub]/[anonymized-repository-name].git',str(ROOT)])
else: git(['fetch','origin','--prune'],cwd=ROOT)
git(['checkout','main'],cwd=ROOT); git(['pull','--ff-only','origin','main'],cwd=ROOT)
HEAD=git(['rev-parse','HEAD'],cwd=ROOT,capture=True).stdout.strip(); print('main HEAD',HEAD)


In [ ]:
# 2. Fail-closed #41 / #44 provenance preflight.
import hashlib, json, re

AUTHORITY=ROOT/'docs/DEVIATION_2026-08-17_CONFIRMATORY_MEMBERSHIP.md'
COUNTS=ROOT/'results/t15/t15_confirmatory_bundle.counts.json'
REPORT=ROOT/'results/t15/confirmatory_reliability.json'
OUT=ROOT/'results/t14/retained_roles.json'
RECEIPT=ROOT/'results/t14/T14_EXECUTION_RECEIPT.json'

PR44_MERGE='3848b6ccb49bd37ccdb47d127aca688ffda60c11'
PR41_MERGE='fb01dfbd8247448e494136edc3b50a8e3e7d8a14'
REVIEWED_COUNTS_SHA='6fc68c91539c5f3c2625d089f2301fd1ada43847c5253de629e479f4158dd694'
REVIEWED_REPORT_SHA='c95383683b09f0490c6f76c485a1756c7458bb427d5eb5c39adabd138edcdaed'

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):
            h.update(b)
    return h.hexdigest()

# Require both change-control dependency and accepted T15/T16 PR on main.
for name, commit in [('#44', PR44_MERGE), ('#41', PR41_MERGE)]:
    if git(
        ['merge-base','--is-ancestor',commit,'HEAD'],
        cwd=ROOT, check=False
    ).returncode != 0:
        raise RuntimeError(f'main does not contain merged PR {name}')

if not COUNTS.exists() or not REPORT.exists():
    raise RuntimeError('#41 T15/T16 artifacts are not present on main')

# Bind to the independently reviewed counts and report bytes.
assert sha256(COUNTS)==REVIEWED_COUNTS_SHA, \
    'reviewed #41 counts changed; stop and re-review'
assert sha256(REPORT)==REVIEWED_REPORT_SHA, \
    'reviewed #41 report changed; stop and re-review'

counts=json.loads(COUNTS.read_text())
report=json.loads(REPORT.read_text())
prov=counts['provenance']
rp=report['input_provenance']

# Require the accepted clean-tree canonical T15/T16 record.
assert report['source_git_dirty'] is False, \
    'canonical T15/T16 report is not a clean-tree production result'

assert report['source_git_sha'] == \
    '32a787d604e530ca3d1d8f32b8e8a859aedd83db', \
    'canonical T15/T16 generating commit changed; stop and re-review'

# Frozen T14/T15/T16 construction.
assert (
    prov['membership']=='label_independent_technical_validity'
    and prov['block_index']==16
    and prov['pool']=='ALL_RESPONSE_TOKENS'
    and prov['arm']=='USER_TRANSLATED_LU'
)

assert (
    report['membership']=='label_independent_technical_validity'
    and report['n_retained_roles']==275
    and rp['counts_sha256']==REVIEWED_COUNTS_SHA
)

assert (
    rp['eligible_outputs']=={'C80-A':21998,'C80-B':22000}
    and prov['t12_manifest_sha256']==rp['t12_manifest_sha256']
)

assert re.fullmatch(
    r'[0-9a-f]{40}',
    rp['bundle_fetched_at_hf_revision']
)

assert sum(map(int,counts['a'].values()))==21998
assert sum(map(int,counts['b'].values()))==22000
assert set(counts['a'])==set(counts['b'])
assert len(counts['a'])==275
assert min(map(int,counts['a'].values()))>=10
assert min(map(int,counts['b'].values()))>=10

print('PASS: accepted #41 counts/report are bound correctly')
print('counts SHA', sha256(COUNTS))
print('HF bundle revision', rp['bundle_fetched_at_hf_revision'])


In [ ]:
# 3. Run T14 unit tests, then execute the production freeze.
subprocess.check_call([sys.executable,'-m','pytest','-q','tests/test_t14_confirmatory_role_freeze.py'],cwd=ROOT)
OUT.parent.mkdir(parents=True,exist_ok=True)
subprocess.check_call([sys.executable,'tools/freeze_t14_confirmatory_roles.py','--counts',str(COUNTS.relative_to(ROOT)),'--authority',str(AUTHORITY.relative_to(ROOT)),'--out',str(OUT.relative_to(ROOT))],cwd=ROOT)
t14=json.loads(OUT.read_text())
assert t14['status']=='FROZEN' and t14['retained_role_count']==275 and len(t14['retained_roles'])==275
assert set(t14['retained_roles'])==set(counts['a'])==set(counts['b'])
assert t14['eligible_outputs']=={'C80-A':21998,'C80-B':22000} and t14['counts_sha256']==REVIEWED_COUNTS_SHA
print('PASS: T14 frozen',OUT)


In [ ]:
# 4. Write an execution receipt binding T14 to the accepted T15/T16 artifact.
from datetime import datetime, timezone
receipt={'schema_version':'t14-execution-receipt/1.0','task':'T14','status':'FROZEN','created_utc':datetime.now(timezone.utc).isoformat(),'source_git_sha':git(['rev-parse','HEAD'],cwd=ROOT,capture=True).stdout.strip(),'membership_authority':str(AUTHORITY.relative_to(ROOT)),'membership_authority_sha256':sha256(AUTHORITY),'t15_t16_counts':str(COUNTS.relative_to(ROOT)),'t15_t16_counts_sha256':sha256(COUNTS),'t15_t16_report':str(REPORT.relative_to(ROOT)),'t15_t16_report_sha256':sha256(REPORT),'bundle_fetched_at_hf_revision':rp['bundle_fetched_at_hf_revision'],'t12_manifest_sha256':prov['t12_manifest_sha256'],'retained_role_manifest':str(OUT.relative_to(ROOT)),'retained_role_manifest_sha256':sha256(OUT),'retained_role_count':275,'eligible_outputs':{'C80-A':21998,'C80-B':22000},'claim_scope':'technical-validity retained-role set; does not assert successful behavioural role enactment'}
RECEIPT.write_text(json.dumps(receipt,indent=2)+'\n'); print(json.dumps(receipt,indent=2))


In [ ]:
# 5. Review bundle. Do not auto-push; inspect retention/provenance before committing.
subprocess.run(['git','-C',str(ROOT),'status','--short'])
print('review:',OUT.relative_to(ROOT),sha256(OUT)); print('review:',RECEIPT.relative_to(ROOT),sha256(RECEIPT))
print('After review: git add results/t14/retained_roles.json results/t14/T14_EXECUTION_RECEIPT.json')
